# Group 3 | Seasonal Climatology

**Predicting Drought Before the Rains Fail**

District-level severe meteorological drought, Tanzania (1981–2025).

## 4. Seasonal Climatology: mathematical specification

**Objective.** Estimate district- and calendar-month-specific severe-drought probabilities from earlier historical observations.

$$\boxed{\hat p_{d,m}=\frac{D_{d,m}}{N_{d,m}}}$$

$D_{d,m}$ counts previous years in which district $d$ experienced severe SPI-3 drought in calendar month $m$. $N_{d,m}$ counts previous years with valid SPI-3 for that district/month. No weights, smoothing, satellite predictors or regression parameters are applied.

A forecast is undefined if $N_{d,m}=0$. Historical observations at or after the forecast issue date must never enter $D$ or $N$.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "outputs"
for name in ["figures", "maps", "metrics", "predictions", "tables"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

import pandas as pd
from IPython.display import display
from group3_climatology.data import load_data
from group3_climatology.baseline import forecast_table, climatology_snapshot, inspect_forecast
from group3_climatology.config import INPUT, SPI_FIELD

data=load_data(INPUT)
forecasts=forecast_table(data)
example=inspect_forecast(forecasts,"Nanyumbu","2020-04-01",3)
display(example)

,district_id,region,district,issue_month,target_month,horizon_months,calendar_month,historical_severe_years,historical_valid_years,predicted_probability,observed_spi3,observed_severe,severe_onset,latest_used_month_upper_bound
0,0905,Mtwara,Nanyumbu,2020-01-01,2020-04-01,3,4,4.0,39.0,0.102564,-1.768952,1,1,2019-04-01


### Worked calculation: Nanyumbu, April 2020

The forecast issued in January 2020 uses earlier April observations only.

$$\hat p_{\mathrm{Nanyumbu,Apr}}=\frac{4}{39}=0.10256\approx10.26\%.$$

The observed April 2020 SPI-3 was below $-1.5$, so the event subsequently met the severe-drought definition. This example is one forecast verification, not a model-wide accuracy estimate.

In [2]:
row=example.iloc[0]
D=int(row.historical_severe_years); N=int(row.historical_valid_years)
print(f"Historical count D = {D}; valid observations N = {N}")
print(f"D/N = {D}/{N} = {D/N:.8f} ({100*D/N:.2f}%)")
print(f"Issue: {row.issue_month:%Y-%m}; target: {row.target_month:%Y-%m}; observed SPI-3: {row.observed_spi3:.3f}")
assert D==4 and N==39
assert abs(row.predicted_probability-D/N)<1e-12
history=data[(data.district.eq("Nanyumbu")) & (data.month.dt.month==4) &
             (data.month.lt(pd.Timestamp("2020-01-01"))) & data[SPI_FIELD].notna()].copy()
history['historical_severe']=history[SPI_FIELD].le(-1.5).astype(int)
display(history[["month",SPI_FIELD,"historical_severe"]].tail(12))
assert int(history.historical_severe.sum())==D and len(history)==N

Historical count D = 4; valid observations N = 39
D/N = 4/39 = 0.10256410 (10.26%)
Issue: 2020-01; target: 2020-04; observed SPI-3: -1.769


,month,spi3_gamma_ref1981_2010,historical_severe
30027,2008-04-01,-0.078362,0
30039,2009-04-01,-0.938592,0
30051,2010-04-01,0.192518,0
30063,2011-04-01,1.544626,0
30075,2012-04-01,-0.515850,0
30087,2013-04-01,-1.990634,1
30099,2014-04-01,1.158860,0
30111,2015-04-01,-0.467202,0
30123,2016-04-01,0.463614,0
30135,2017-04-01,1.102073,0


### Historical probability register

A district-month table summarizes the historical information available through December 2019, before the selected 2020–2022 case-study period.

In [3]:
snapshot=climatology_snapshot(data,cutoff="2019-12-01")
display(snapshot.head(12))
print("District–calendar-month probabilities:",len(snapshot))
snapshot.to_csv(OUT / "predictions" / "district_month_climatology_as_of_2019.csv",index=False)

,district_id,region,district,calendar_month,D,N,probability
0,0202,Arusha,Arumeru,1,2,38,0.052632
1,0202,Arusha,Arumeru,2,3,38,0.078947
2,0202,Arusha,Arumeru,3,5,39,0.128205
3,0202,Arusha,Arumeru,4,1,39,0.025641
4,0202,Arusha,Arumeru,5,4,39,0.102564
5,0202,Arusha,Arumeru,6,5,39,0.128205
6,0202,Arusha,Arumeru,7,2,39,0.051282
7,0202,Arusha,Arumeru,8,2,39,0.051282
8,0202,Arusha,Arumeru,9,0,39,0.000000
9,0202,Arusha,Arumeru,10,0,39,0.000000


District–calendar-month probabilities: 1800


### Model assumptions and limitations

Seasonal Climatology assumes that historical frequency is informative about future seasonal risk. Its forecasts do not respond to current rainfall, soil moisture, NDVI or evolving drought conditions. Probabilities may be poorly calibrated if climate patterns change. A probability is not automatically an alert; warnings require a separately specified decision threshold.